In [2]:
import pandas as pd

df = pd.read_csv("email.csv")

# clean
df["Category"] = df["Category"].str.lower().str.strip()
df["Message"] = df["Message"].fillna("")

# keep valid rows
df = df[df["Category"].isin(["ham", "spam"])]

# Y (labels)
df["y"] = df["Category"].map({"ham": 0, "spam": 1})

X_text = df["Message"]
y = df["y"]


In [3]:
df

,Category,Message,y
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0
...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,1
5568,ham,Will ü b going to esplanade fr home?,0
5569,ham,"Pity, * was in mood for that. So...any other s...",0
5570,ham,The guy did some bitching but I acted like i'd...,0


In [5]:
# Train / Test split
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [6]:
# Convert text → numbers (TF-IDF)
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english", max_features=10000)

X_train = vectorizer.fit_transform(X_train_text)
X_test  = vectorizer.transform(X_test_text)


In [8]:
# Train model (Gradient Descent)
from sklearn.linear_model import SGDClassifier

model = SGDClassifier(loss="log_loss", random_state=42)
# Input → Output → Loss → Gradient Descent → Update weights
model.fit(X_train, y_train)


,"loss loss: {'hinge', 'log_loss', 'modified_huber', 'squared_hinge', 'perceptron', 'squared_error', 'huber', 'epsilon_insensitive', 'squared_epsilon_insensitive'}, default='hinge'The loss function to be used.- 'hinge' gives a linear SVM.- 'log_loss' gives logistic regression, a probabilistic classifier.- 'modified_huber' is another smooth loss that brings tolerance to outliers as well as probability estimates.- 'squared_hinge' is like hinge but is quadratically penalized.- 'perceptron' is the linear loss used by the perceptron algorithm.- The other losses, 'squared_error', 'huber', 'epsilon_insensitive' and 'squared_epsilon_insensitive' are designed for regression but can be useful in classification as well; see :class:`~sklearn.linear_model.SGDRegressor` for a description.More details about the losses formulas can be found in the :ref:`User Guide` and you can find a visualisation of the lossfunctions in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_loss_functions.py`.",'log_loss'
,"penalty penalty: {'l2', 'l1', 'elasticnet', None}, default='l2'The penalty (aka regularization term) to be used. Defaults to 'l2'which is the standard regularizer for linear SVM models. 'l1' and'elasticnet' might bring sparsity to the model (feature selection)not achievable with 'l2'. No penalty is added when set to `None`.You can see a visualisation of the penalties in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_penalties.py`.",'l2'
,"alpha alpha: float, default=0.0001Constant that multiplies the regularization term. The higher thevalue, the stronger the regularization. Also used to compute thelearning rate when `learning_rate` is set to 'optimal'.Values must be in the range `[0.0, inf)`.",0.0001
,"l1_ratio l1_ratio: float, default=0.15The Elastic Net mixing parameter, with 0 <= l1_ratio <= 1.l1_ratio=0 corresponds to L2 penalty, l1_ratio=1 to L1.Only used if `penalty` is 'elasticnet'.Values must be in the range `[0.0, 1.0]` or can be `None` if`penalty` is not `elasticnet`... versionchanged:: 1.7 `l1_ratio` can be `None` when `penalty` is not ""elasticnet"".",0.15
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If False, thedata is assumed to be already centered.",True
,"max_iter max_iter: int, default=1000The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the ``fit`` method, and not the:meth:`partial_fit` method.Values must be in the range `[1, inf)`... versionadded:: 0.19",1000
,"tol tol: float or None, default=1e-3The stopping criterion. If it is not None, training will stopwhen (loss > best_loss - tol) for ``n_iter_no_change`` consecutiveepochs.Convergence is checked against the training loss or thevalidation loss depending on the `early_stopping` parameter.Values must be in the range `[0.0, inf)`... versionadded:: 0.19",0.001
,"shuffle shuffle: bool, default=TrueWhether or not the training data should be shuffled after each epoch.",True
,"verbose verbose: int, default=0The verbosity level.Values must be in the range `[0, inf)`.",0
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-insensitive loss functions; only if `loss` is'huber', 'epsilon_insensitive', or 'squared_epsilon_insensitive'.For 'huber', determines the threshold at which it becomes lessimportant to get the prediction exactly right.For epsilon-insensitive, any differences between the current predictionand the correct label are ignored if they are less than this threshold.Values must be in the range `[0.0, inf)`.",0.1
,"n_jobs n_jobs: int, default=NoneThe number of CPUs to use to do the OVA (One Versus All, formulti-class problems) computation.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [9]:
# Predict on X_test
y_pred = model.predict(X_test)          # final class (0/1)
y_prob = model.predict_proba(X_test)    # probabilities

In [11]:
y_prob

array([[0.98015791, 0.01984209],
       [0.98938555, 0.01061445],
       [0.98337581, 0.01662419],
       ...,
       [0.93442904, 0.06557096],
       [0.94691748, 0.05308252],
       [0.97215752, 0.02784248]], shape=(1115, 2))

In [12]:
# Calculate loss on X_test
from sklearn.metrics import log_loss, accuracy_score

test_loss = log_loss(y_test, y_prob)
test_accuracy = accuracy_score(y_test, y_pred)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)


Test Loss: 0.10084271683114471
Test Accuracy: 0.9775784753363229


In [13]:
# See some real predictions

results = pd.DataFrame({
    "Message": X_test_text.values[:10],
    "Actual": y_test.values[:10],
    "Predicted": y_pred[:10],
    "Spam_Probability": y_prob[:10, 1]
})

results


,Message,Actual,Predicted,Spam_Probability
0,No need to buy lunch for me.. I eat maggi mee..,0,0,0.019842
1,Ok im not sure what time i finish tomorrow but...,0,0,0.010614
2,Waiting in e car 4 my mum lor. U leh? Reach ho...,0,0,0.016624
3,"You have won ?1,000 cash or a ?2,000 prize! To...",1,1,0.892236
4,If you r @ home then come down within 5 min,0,0,0.044249
5,No need lar. Jus testing e phone card. Dunno n...,0,0,0.032869
6,Wot about on wed nite I am 3 then but only til 9!,0,0,0.044403
7,Hey you gave them your photo when you register...,0,0,0.041693
8,"Hello, my boytoy! I made it home and my consta...",0,0,0.007116
9,Eh u remember how 2 spell his name... Yes i di...,0,0,0.025536
